# Fine-tune the Tunisian Arabizi sentiment model (Model B)

Trains a **TUNIZI** specialist (Model B) on a free GPU and produces the per-language
macro-F1 table comparing the multilingual baseline (Model A) against the fine-tuned
model. The accuracy jump on `aeb-latn` is the headline result for the report.

Runs on **Kaggle** (GPU T4) or **Google Colab**.

**Before you run anything:**
- Enable a **GPU** — Kaggle: *Session options -> Accelerator -> GPU T4*. Colab: *Runtime -> Change runtime type -> T4 GPU*.
- Enable **Internet** — Kaggle: *Session options -> Internet -> On* (needed for the clone, pip install, and model download).
- Have the **`tunizi.csv`** ready (columns `text,label`). Kaggle: add it via *+ Add Input*. Colab: upload it in step 2.

## 1. Get the code and install the training extras

Clones the repo and installs the `nlp` + `train` dependencies. It also removes
`torchvision`/`torchaudio`: text classification does not need them, and the versions
pre-installed on hosted GPUs clash with the pinned `torch`, which otherwise crashes
the transformers import.

If your repo is **private**, `git clone` will fail here — make it public briefly, or
upload the `app/` folder manually.

In [ ]:
!git clone https://github.com/firas191/Community-Management.git 2>/dev/null || echo 'repo already cloned'
%cd Community-Management
!pip install -q -e ".[nlp,train]"
# Text-only training: drop vision/audio so they cannot clash with the pinned torch.
!pip uninstall -y torchvision torchaudio 2>/dev/null || true
print('\ninstall done - the red pip "dependency conflict" lines above are about the'
      " host's own packages, not this project.")

## 2. Configuration

Edit `DATA` to point at your `tunizi.csv`, then run. This cell also forces the
PyTorch-only backend (`USE_TF=0`, `USE_FLAX=0`), checks the GPU, and confirms the
dataset is reachable before you spend time training.

- **Kaggle**: after *+ Add Input*, the path looks like `/kaggle/input/<dataset-slug>/tunizi.csv` (check the left file browser for the exact slug).
- **Colab**: run `from google.colab import files; files.upload()` in a scratch cell, then set `DATA = 'tunizi.csv'`.

In [ ]:
import os

# Use only the PyTorch backend (hosted GPUs ship a TF/Flax that clashes with the
# pinned protobuf; transformers auto-imports whatever backends it detects).
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'

# ---- EDIT THIS to your tunizi.csv path ----
DATA = '/kaggle/input/datasets/firas19/tunizi-csv/tunizi.csv'
# -------------------------------------------
BASE_MODEL = 'cardiffnlp/twitter-xlm-roberta-base'            # base for fine-tuning (Model B)
BASELINE   = 'cardiffnlp/twitter-xlm-roberta-base-sentiment'  # already-trained Model A, for comparison
OUT_DIR    = '/kaggle/working/models/arabizi'                 # where Model B is saved
TEST       = '/kaggle/working/test.csv'                       # held-out split, written in step 4

import torch
print('GPU available:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - enable it in Session options')
assert os.path.exists(DATA), f'CSV not found at {DATA!r} - fix DATA above'
print('dataset OK:', DATA)

## 3. Fine-tune (stratified 70/10/20, lr 2e-5, early stopping on macro-F1)

Downloads the base model once (~1 GB), then trains up to 4 epochs with class weights
and early stopping. On a T4 this is a few minutes. The final line reports the
held-out **test** accuracy / macro-F1 and saves the model to `OUT_DIR`.

Note: TUNIZI is binary (positive/negative), so the 3-class macro-F1 is deflated by the
empty `neutral` class - read **accuracy** and the neg/pos F1s as the real signal.

In [ ]:
!USE_TF=0 USE_FLAX=0 python -m app.nlp.training.finetune_tunizi --csv {DATA} --base-model {BASE_MODEL} --output-dir {OUT_DIR} --epochs 4

## 4. Rebuild the held-out test split

Reproduces the exact test fold used in training (same seed, deterministic) and writes
it to `test.csv`. Both models are then scored on **these rows only**, which Model B
never trained on - so the comparison is honest.

In [ ]:
import csv
from app.nlp.training.data import normalize_label, stratified_split

rows = []
with open(DATA, encoding='utf-8') as f:
    for r in csv.DictReader(f):
        lab = normalize_label(r.get('label'))
        txt = (r.get('text') or '').strip()
        if lab and txt:
            rows.append({'text': txt, 'label': lab})

split = stratified_split(rows)  # seed=42, identical to training
with open(TEST, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['text', 'label'])
    for r in split.test:
        w.writerow([r['text'], r['label']])
print('held-out test rows:', len(split.test), '->', TEST)

## 5. Before / after: Model A vs the fine-tuned Model B

Runs the same held-out set through the multilingual baseline (Model A, which never saw
TUNIZI) and your fine-tuned model (Model B). The per-language table prints macro-F1 and
accuracy; the **`aeb-latn` accuracy delta** is the headline number for the report.

In [ ]:
print('=== Model A (baseline, never saw TUNIZI) ===')
!USE_TF=0 USE_FLAX=0 python -m app.nlp.training.evaluate --csv {TEST} --model {BASELINE}
print('\n=== Model B (fine-tuned) ===')
!USE_TF=0 USE_FLAX=0 python -m app.nlp.training.evaluate --csv {TEST} --model {OUT_DIR}

## 6. Download the trained model

Zips `OUT_DIR` into `arabizi_model.zip`. On **Kaggle**, grab it from the **Output**
panel on the right (or the link below). On **Colab**, use
`from google.colab import files; files.download('/kaggle/working/arabizi_model.zip')`
(adjust the path to your `OUT_DIR`).

Then unzip it where your Docker `api`/`worker` can see it, set `ARABIZI_MODEL` to that
path, and rebuild - Arabizi comments will route through Model B.

In [ ]:
import shutil

zip_path = shutil.make_archive('/kaggle/working/arabizi_model', 'zip', OUT_DIR)
print('zipped ->', zip_path)
from IPython.display import FileLink
FileLink('/kaggle/working/arabizi_model.zip')